In [1]:
# Parallel Reduction Multiplication using CUDA in Python (Numba)

import numpy as np
from numba import cuda, float32
import math
import time

# CUDA Kernel for Parallel Reduction Multiplication
@cuda.jit
def reduction_multiply(input_array, output_array):

    # Shared memory
    shared_data = cuda.shared.array(256, dtype=float32)

    tid = cuda.threadIdx.x
    idx = cuda.grid(1)

    # Load data into shared memory
    if idx < input_array.size:
        shared_data[tid] = input_array[idx]
    else:
        shared_data[tid] = 1.0

    # Synchronize threads
    cuda.syncthreads()

    # Parallel Reduction Multiplication
    stride = cuda.blockDim.x // 2

    while stride > 0:

        if tid < stride:
            shared_data[tid] *= shared_data[tid + stride]

        cuda.syncthreads()

        stride //= 2

    # Store block result
    if tid == 0:
        output_array[cuda.blockIdx.x] = shared_data[0]


# Driver Code
if __name__ == "__main__":

    N = 256

    # Input Array
    input_data = np.arange(1, N + 1).astype(np.float32)

    print("Input Array:")
    print(input_data[:10])

    # Output array for block results
    output_size = math.ceil(N / 256)
    output_data = np.zeros(output_size, dtype=np.float32)

    # Copy to GPU
    d_input = cuda.to_device(input_data)
    d_output = cuda.to_device(output_data)

    threads_per_block = 256
    blocks_per_grid = output_size

    start = time.time()

    # Launch Kernel
    reduction_multiply[blocks_per_grid, threads_per_block](d_input, d_output)

    cuda.synchronize()

    end = time.time()

    # Copy result back
    partial_result = d_output.copy_to_host()

    # Final multiplication on CPU
    final_result = np.prod(partial_result)

    print("\nPartial Block Products:")
    print(partial_result)

    print("\nFinal Product:")
    print(final_result)

    print("\nExecution Time:", end - start, "seconds")

Input Array:
[ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10.]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))



Partial Block Products:
[inf]

Final Product:
inf

Execution Time: 2.142974853515625 seconds
